# Phase 0b — Python cross-check of `log q(config)`

Independent numpy implementation (`PEPS.proposal_logprob`, added to
[`peps.py`](../peps.py)) of the same deterministic proposal log-probability.
We reproduce the two controls (`Σq = 1`, `C = 1` at exact `D`) and write
`log q` for a fixed set of configs to `../jl/icfo_intern/results/crosscheck_py.csv`
so the Julia Phase-0 notebook can compare (must agree ≤ 1e-8).

**Environment:** the committed `icfo_env/` is a Windows venv; recreate on Linux:
`python3 -m venv .venv && source .venv/bin/activate && pip install numpy scipy opt_einsum`.


In [1]:
import os, numpy as np, itertools
from peps import PEPS

betac = np.log(1 + np.sqrt(2)) / 2
Lx, Ly = 3, 4

def config_from_int(i, Lx, Ly):
    # column-major bit order — identical to tools/tnmh_tools.jl config_from_int
    cfg = np.zeros((Lx, Ly), dtype=int)
    for k in range(Lx * Ly):
        x = k % Lx
        y = k // Lx
        cfg[x, y] = (i >> k) & 1
    return cfg

def total_energy(cfg, J=1.0):
    s = np.where(cfg == 0, 1, -1)
    return -J * (np.sum(s[:, :-1] * s[:, 1:]) + np.sum(s[:-1, :] * s[1:, :]))

print("ready. beta_c =", round(betac, 6))


ready. beta_c = 0.440687


## Controls A & B (Σq = 1 ; C = 1 at exact D)

In [2]:
for D in (2, 4):
    ising = PEPS.create_ising_2d(Lx, Ly, beta=betac)
    sq = 0.0; Es = []; logqs = []
    for bits in itertools.product([0, 1], repeat=Lx * Ly):
        cfg = config_from_int(sum(b << k for k, b in enumerate(bits)), Lx, Ly)
        lq = ising.proposal_logprob(cfg, D)
        sq += np.exp(lq); Es.append(total_energy(cfg)); logqs.append(lq)
    Es = np.array(Es); logqs = np.array(logqs)
    logZ = np.log(np.sum(np.exp(-betac * Es)))
    w = np.exp((-betac * Es - logZ) - logqs)
    print(f"D={D}:  sum_q={sq:.10f}   C={w.max():.8f}")


D=2:  sum_q=1.0000000000   C=1.00128585
D=4:  sum_q=1.0000000000   C=1.00000000


## Write `crosscheck_py.csv` for the Julia comparison

In [3]:
out_dir = os.path.join("..", "jl", "icfo_intern", "results")
os.makedirs(out_dir, exist_ok=True)
ising = PEPS.create_ising_2d(Lx, Ly, beta=betac)
with open(os.path.join(out_dir, "crosscheck_py.csv"), "w") as f:
    f.write("id,logq\n")
    for i in range(32):
        cfg = config_from_int(i, Lx, Ly)
        f.write(f"{i},{ising.proposal_logprob(cfg, 2):.15e}\n")
print("wrote", os.path.join(out_dir, "crosscheck_py.csv"))
print("Now run phase0_infrastructure.ipynb (Julia) — the cross-check cell will assert ≤1e-8.")


wrote ../jl/icfo_intern/results/crosscheck_py.csv
Now run phase0_infrastructure.ipynb (Julia) — the cross-check cell will assert ≤1e-8.
